# EN Llama 3 Baseline and QLoRA

Основной Colab-ноутбук для английской части эксперимента. В нем последовательно запускаются baseline-генерация, QLoRA-адаптация модели `meta-llama/Meta-Llama-3-8B-Instruct`, генерация через адаптер и автоматическая стилометрическая оценка.

## 1. Подготовка репозитория

Эту ячейку нужно выполнить первой. Если ноутбук открыт в чистом Colab runtime, она скачает публичный репозиторий с GitHub и перейдет в корень проекта. Если вы уже находитесь в локальной копии репозитория, ячейка просто проверит наличие нужных папок и данных.

In [ ]:
# Подготовка окружения Colab.
# Если ноутбук открыт не из корня репозитория, эта ячейка скачает публичный GitHub-репозиторий
# и переключит рабочую директорию на /content/llama_test.
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/chipalgash/llama_test.git"
REPO_DIR = Path("/content/llama_test")

if not Path("src").exists():
    if not REPO_DIR.exists():
        print(f"Cloning {REPO_URL} -> {REPO_DIR}")
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
else:
    print("Already in project root; using current working directory.")

print("Current directory:", Path.cwd())
assert Path("src").exists(), "src/ not found. Check repository clone or %cd to project root."
assert Path("data/processed/en/eval_payload.jsonl").exists(), "EN eval payload is missing."
assert Path("data/processed/ru/eval_payload.jsonl").exists(), "RU eval payload is missing."
print("Repository is ready.")

## 2. Установка зависимостей

Устанавливаются библиотеки для загрузки Llama 3, 4-bit quantization, LoRA/QLoRA-обучения, работы с датасетами и статистических тестов. Для этого ноутбука нужен GPU runtime.

In [ ]:
# Установка библиотек для инференса, QLoRA-обучения и статистической оценки.
# Запускайте эту ячейку один раз после старта Colab runtime.
!pip install -q transformers accelerate peft trl bitsandbytes datasets scipy

## 3. Авторизация Hugging Face и проверка данных

Введите `HF_TOKEN` от аккаунта Hugging Face. У токена должен быть доступ к модели `meta-llama/Meta-Llama-3-8B-Instruct`; иначе загрузка весов остановится на этапе baseline или обучения.

In [ ]:
# Авторизация Hugging Face.
# Нужен токен HF_TOKEN с доступом к meta-llama/Meta-Llama-3-8B-Instruct.
# Токен вводится скрыто и сохраняется только в переменную окружения текущей Colab-сессии.
import getpass
import os
from pathlib import Path

assert Path('src').exists(), 'Run %cd to the project root before continuing.'
assert Path('data/processed/en/eval_payload.jsonl').exists(), 'EN eval payload is missing.'

if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass.getpass('HF_TOKEN: ')

!huggingface-cli whoami

## 4. Baseline-генерация

Эта команда генерирует тексты базовой Llama 3 без дообучения. Результат сохраняется в `outputs/en/baseline_generations.csv` и дальше используется как контрольная группа.

In [ ]:
# Генерация английского baseline: модель без LoRA-адаптера.
# --max-samples 100 означает, что берутся первые 100 примеров из eval payload.
!PYTHONPATH=$PWD python -m src.generation.generate_baseline \
  --language en \
  --eval-jsonl data/processed/en/eval_payload.jsonl \
  --output-csv outputs/en/baseline_generations.csv \
  --max-samples 100

## 5. QLoRA-обучение

Здесь обучается компактный LoRA-адаптер поверх замороженной 4-bit Llama 3. Полные веса модели не сохраняются; результатом является папка `adapters/en_lora_adapter`. Этот этап требует GPU.

In [ ]:
# Обучение английского QLoRA-адаптера.
# Параметры выборки ограничены, чтобы эксперимент укладывался в Colab GPU runtime.
!PYTHONPATH=$PWD python -m src.training.train_qlora \
  --train-jsonl data/processed/en/train_payload.jsonl \
  --eval-jsonl data/processed/en/eval_payload.jsonl \
  --adapter-dir adapters/en_lora_adapter \
  --max-train-samples 500 \
  --max-eval-samples 100

## 6. Генерация через fine-tuned адаптер

После обучения команда загружает ту же базовую Llama 3 и подключает сохраненный LoRA-адаптер. Результат сохраняется в `outputs/en/finetuned_generations.csv`. Для честного сравнения желательно использовать то же число примеров, что и в baseline.

In [ ]:
# Генерация английских текстов через обученный LoRA-адаптер.
# Если нужно быстро проверить качество, можно временно уменьшить --max-samples, например до 25.
!PYTHONPATH=$PWD python -m src.generation.generate_with_adapter \
  --language en \
  --eval-jsonl data/processed/en/eval_payload.jsonl \
  --adapter-dir adapters/en_lora_adapter \
  --output-csv outputs/en/finetuned_generations.csv \
  --max-samples 100

## 7. Стилометрия и статистика

Финальный шаг собирает признаки для human, baseline и fine-tuned текстов, считает сводные метрики и статистические сравнения. Основные файлы результата: `outputs/en/stylometric_features.csv` и `outputs/en/metric_summary.csv`.

In [ ]:
# Автоматическая оценка английского прогона.
# Скрипт сравнивает человеческие тексты, baseline-генерации и fine-tuned генерации.
!PYTHONPATH=$PWD python -m src.evaluation.run_stylometry \
  --language en \
  --human-jsonl data/processed/en/eval_payload.jsonl \
  --baseline-csv outputs/en/baseline_generations.csv \
  --finetuned-csv outputs/en/finetuned_generations.csv \
  --features-csv outputs/en/stylometric_features.csv \
  --summary-csv outputs/en/metric_summary.csv